In [ ]:
import logging
import os
import numpy as np
import open3d as o3d
from PIL import Image
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from marmopose.version import __version__ as marmopose_version
from marmopose.config import Config

from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN, HDBSCAN
from sklearn.preprocessing import StandardScaler
import umap

from skimage.segmentation import watershed
from skimage.feature import peak_local_max
from scipy.ndimage import gaussian_filter
from scipy.interpolate import RegularGridInterpolator

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(name)s - %(message)s')
logger = logging.getLogger(__name__)

logger.info(f'MarmoPose version: {marmopose_version}')

from marmopose.utils.data_io import load_points_3d_h5


In [ ]:
os.chdir('..')
config_path = '../configs/default.yaml'

config = Config(
    config_path=config_path,
    
    n_tracks=1,
    project='../demos/single',
)
os.chdir('umap')


In [ ]:
SRC_DIR = "/scratch/VideoTracking/Videos/Test3.5"
points_3d = load_points_3d_h5(os.path.join(SRC_DIR,"Output/points_3d/optimized.h5"))
idx_spinemid = config.animal['bodyparts'].index('spinemid')
idx_tailbase = config.animal['bodyparts'].index('tailbase')
idx_neck = config.animal['bodyparts'].index('neck')
points_3d_ = points_3d - points_3d[:,:,[idx_spinemid],:]
vector_body = points_3d_[:,:,[idx_neck],:] - points_3d_[:,:,[idx_tailbase],:]
magnitudes_vector_body_xy = np.sqrt(vector_body[0,:,0,0]**2 + vector_body[0,:,0, 1]**2)[:,np.newaxis,np.newaxis]
rotation_matrices = np.array([[vector_body[0,:,0,0],vector_body[0,:,0,1]],[-vector_body[0,:,0,1],vector_body[0,:,0,0]]]).transpose((2,0,1))/magnitudes_vector_body_xy
rotated_points_3d = np.copy(points_3d_)[0,:,:,:]
rotated_points_3d[:,:,:2] = np.einsum('ikl,ijl -> ijk', rotation_matrices, rotated_points_3d[:,:,:2])
rotated_points_3d_ = rotated_points_3d.reshape((-1,48))
points_3d_velocity = points_3d[0,1:,:,:] - points_3d[0,:-1,:,:]
rotated_points_3d_velocity_3d = np.copy(points_3d_velocity)
rotated_points_3d_velocity_3d[:,:,:2] = np.einsum('ikl,ijl -> ijk', rotation_matrices[:-1,:,:], rotated_points_3d_velocity_3d[:,:,:2])
rotated_points_3d_velocity = rotated_points_3d_velocity_3d.reshape(-1,48)
# inputs = np.concatenate((rotated_points_3d_[:-1,:],rotated_points_3d_velocity), axis = 1)
inputs = rotated_points_3d_

In [ ]:
scaler = StandardScaler()
normalized_inputs = scaler.fit_transform(inputs)
pca = PCA(n_components=0.95)
inputs_pca = pca.fit_transform(normalized_inputs)
print(f"Number of principal components retained: {inputs_pca.shape[1]}")
print(pca.explained_variance_ratio_)

In [ ]:
X_umap = umap.UMAP(
    n_neighbors=100,
    min_dist=0,
    n_components=2,
    random_state=42
).fit_transform(inputs_pca)

fig = plt.figure()
ax = fig.add_subplot(111)
ax.scatter(X_umap[:,0],X_umap[:,1])
fig.show()


In [ ]:
from sklearn.metrics import silhouette_score

silhouette_scores = []
for i in range(2, 12):
    for j in range(5,30,5):
        clustering = DBSCAN(eps=i, min_samples=j)
        cluster_labels = clustering.fit_predict(X_umap)
        if len(np.unique(cluster_labels)) == 1:
            break 
        silhouette_avg = silhouette_score(X_umap, cluster_labels)
        silhouette_scores.append(silhouette_avg)
        print(f"For eps = {i} and min_samples = {j}, the silhouette score is: {silhouette_avg}")
        np.random.seed(3)
        fig = plt.figure()
        ax = fig.add_subplot(111)
        ax.set_title(f'EPS = {i}, Min_samples = {j}, silhouette score = {silhouette_avg}, ncluster = {len(np.unique(cluster_labels)) + np.min(cluster_labels)}')
        c = np.random.rand(len(np.unique(cluster_labels)),3)

        ax.scatter(X_umap[:,0],X_umap[:,1], c=c[cluster_labels - np.min(cluster_labels)])
        fig.show()

n_clusters_opt = np.argmax(silhouette_scores) + 2

plt.plot(range(2, 2 + len(silhouette_scores)), silhouette_scores, marker='o')
plt.title('Silhouette Scores')
plt.xlabel('Number of clusters')
plt.ylabel('Silhouette Score')
plt.show()


In [ ]:
from sklearn.metrics import silhouette_score
%matplotlib inline
silhouette_scores = []
for i in range(70, 150,10):
    for j in range(2,11,2):
        clustering = HDBSCAN(min_cluster_size=i, min_samples=j)
        cluster_labels = clustering.fit_predict(X_umap)
        if len(np.unique(cluster_labels)) == 1:
            break 
        silhouette_avg = silhouette_score(X_umap, cluster_labels)
        silhouette_scores.append(silhouette_avg)
        print(f"For eps = {i} and min_samples = {j}, the silhouette score is: {silhouette_avg}")
        np.random.seed(3)
        fig = plt.figure()
        ax = fig.add_subplot(111)
        ax.set_title(f'EPS = {i}, Min_samples = {j}, silhouette score = {silhouette_avg}, ncluster = {len(np.unique(cluster_labels)) + np.min(cluster_labels)}')
        c = np.random.rand(len(np.unique(cluster_labels)),3)

        ax.scatter(X_umap[:,0],X_umap[:,1], c=c[cluster_labels - np.min(cluster_labels)])
        fig.show()

n_clusters_opt = np.argmax(silhouette_scores) + 2

plt.plot(range(2, 2 + len(silhouette_scores)), silhouette_scores, marker='o')
plt.title('Silhouette Scores')
plt.xlabel('Number of clusters')
plt.ylabel('Silhouette Score')
plt.show()


In [ ]:
DBscan = DBSCAN(eps=2, min_samples=20)
clusters = DBscan.fit_predict(X_umap)
n_clusters_opt = len(np.unique(clusters)) + np.min(clusters)


In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
np.random.seed(3)
plt.clf()
# Create a 3D scatter plot
fig = plt.figure()
ax = fig.add_subplot(111)
c = np.random.rand(len(np.unique(clusters))  - np.min(clusters),3)
for i in np.unique(clusters):
    cluster_label = 'Noise' if i == -1 else f'Cluster {i}'
    idx_cluster_i = np.nonzero(clusters == i)[0]
    ax.scatter(X_umap[idx_cluster_i,0],X_umap[idx_cluster_i,1], c=c[i - np.min(clusters)], label = cluster_label)
    if i != -1:
        ax.text(np.mean(X_umap[idx_cluster_i,0]),np.mean(X_umap[idx_cluster_i,1]),i)
# ax.set_xlim((None,300))
# ax.legend()
fig.show()

In [ ]:
# DBscan = DBSCAN(eps=7, min_samples=35)
# clusters = DBscan.fit_predict(X_tsne)
# n_clusters_opt = len(np.unique(clusters)) + np.min(clusters)

HDBscan = HDBSCAN(min_cluster_size=140, min_samples=15)
clusters = HDBscan.fit_predict(X_umap)
n_clusters_opt = len(np.unique(clusters)) + np.min(clusters)


In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
np.random.seed(3)
# Create a 3D scatter plot
fig = plt.figure()
ax = fig.add_subplot(111)
c = np.random.rand(len(np.unique(clusters))  - np.min(clusters),3)
for i in np.unique(clusters):
    cluster_label = 'Noise' if i == -1 else f'Cluster {i}'
    idx_cluster_i = np.nonzero(clusters == i)[0]
    ax.scatter(X_umap[idx_cluster_i,0],X_umap[idx_cluster_i,1], c=c[i - np.min(clusters)], label = cluster_label)
    if i != -1:
        ax.text(np.mean(X_umap[idx_cluster_i,0]),np.mean(X_umap[idx_cluster_i,1]),i)
# ax.set_xlim((None,300))
# ax.legend()
fig.show()

In [ ]:


# Create a 2D histogram of the t-SNE points
hist, xedges, yedges = np.histogram2d(X_umap[:, 0], X_umap[:, 1], bins=100, range=[[X_umap[:, 0].min(), X_umap[:, 0].max()], [X_umap[:, 1].min(), X_umap[:, 1].max()]])

# Smooth the histogram with a Gaussian filter
density = gaussian_filter(hist, sigma=3,mode='constant')
# Find local maxima (peaks) in the density map to use as markers
peaks = peak_local_max(density, min_distance=1)
# Label the peaks for watershed
markers = np.zeros_like(density, dtype=int)
markers[peaks[:,0],peaks[:,1]] = np.arange(1, peaks.shape[0]+1)
# Visualize the watershed segments
labels = watershed(-density, markers, mask=density > 0.1 * density.max()) - 1
itp = RegularGridInterpolator((np.linspace(X_umap[:, 0].min(), X_umap[:, 0].max(),labels.shape[0]), np.linspace(X_umap[:, 1].min(), X_umap[:, 1].max(),labels.shape[1])),labels,method='nearest')
clusters = np.round(itp(X_umap)).astype(int)


In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
np.random.seed(3)
# Create a 3D scatter plot
fig = plt.figure()
ax = fig.add_subplot(111)
c = np.random.rand(len(np.unique(clusters))  - np.min(clusters),3)
for i in np.unique(clusters):
    cluster_label = 'Noise' if i == -1 else f'Cluster {i}'
    idx_cluster_i = np.nonzero(clusters == i)[0]
    ax.scatter(X_umap[idx_cluster_i,0],X_umap[idx_cluster_i,1], c=c[i - np.min(clusters)], label = cluster_label)
    if i != -1:
        ax.text(np.mean(X_umap[idx_cluster_i,0]),np.mean(X_umap[idx_cluster_i,1]),i)
# ax.set_xlim((None,300))
# ax.legend()
fig.show()

In [ ]:
np.random.seed(3)
# Select 5 frames for each cluster
idxs = np.array([np.random.choice(np.nonzero(clusters == i)[0],replace=False,size = 5) for i in np.unique(clusters)[-np.min(clusters):]])

In [ ]:
import cv2
frames = np.empty((4,*idxs.shape, 1080, 1920, 3))
# For all cameras
for i in range(1,5):
  vidcap = cv2.VideoCapture(os.path.join(SRC_DIR,f'Output/videos_labeled_2d/output{i}.mp4'))
  # Read first frame
  success,image = vidcap.read()
  j = 0
  while success:
    if not (j % 500):
      print(f'Video {i} frame {j}')
    # Check if id frame corresponds to one of the selected frame
    id_idxs = np.nonzero(idxs == j)
    if id_idxs[0].size:
      # If yes, convert BGR -> RGB and store it in frames array
      image[..., 0:3] = image[..., ::-1]
      frames[i - 1, id_idxs[0][0], id_idxs[1][0]] = image
    success,image = vidcap.read()
    j += 1


In [ ]:
%matplotlib inline
plt.clf()
import matplotlib.colors as mcolors
for i in [16,1,13,14,15]:
# for i in range(np.unique(clusters).size):
    for j in range(5):
        fig, ax = plt.subplots(figsize=(16, 4))
        ax.axis('off')
        norm = mcolors.Normalize(vmin=-20, vmax=20)
        data = rotated_points_3d_velocity_3d[idxs[i,j],:,:].transpose((1,0))
        data = np.concatenate((data,np.mean(data,axis=1,keepdims=True)),axis=1)
        data = np.concatenate((data,np.sqrt(np.einsum("ij,ij->j", data, data))[np.newaxis,:]),axis=0)
        cmap = plt.cm.RdYlBu
        table = ax.table(
            cellText=np.round(data,decimals=2),
            cellColours= cmap(norm(data)),
            rowLabels=['Velocity X','Velocity Y','Velocity Z','Velocity magnitude'],
            colLabels=config.animal['bodyparts'] + ['Mean'],
            loc='center',
            cellLoc='center',
            fontsize = 20
        )
        plt.show()



        fig, ax = plt.subplots(2,2,figsize = (30,15))
        for k in range(4):
            frame = frames[k,i,j]/255
            ax[int(k/2)][k%2].imshow(frame)
            ax[int(k/2)][k%2].set_title(f'Cluster {i}, frame {idxs[i,j]}, camera {k}')
            ax[int(k/2)][k%2].axis('off')
        fig.show()

In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

xlim, ylim, zlim, _ = config.visualization['room_dimensions']

for i in range(n_clusters_opt):
    for j in range(5):
        # Create a 3D scatter plot
        fig = plt.figure()
        ax = fig.add_subplot(111, projection='3d')
        for bodyparts in config.visualization['skeleton'][::-1]:
            idx_bodyparts = []
            for bodypart in bodyparts:
                idx_bodyparts.append(config.animal['bodyparts'].index(bodypart))
            ax.plot(points_3d[0,idxs[i,j],idx_bodyparts,0],points_3d[0,idxs[i,j],idx_bodyparts,1],points_3d[0,idxs[i,j],idx_bodyparts,2], marker = 'o', ms=3)
        ax.set_title(f'Cluster {i}, frame {idxs[i,j]}')
        ax.set_xlim((0,xlim))
        ax.set_ylim((0,ylim))
        ax.set_zlim((0,zlim))

In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

xlim, ylim, zlim, _ = config.visualization['room_dimensions']

for i in range(n_clusters_opt):
    for j in range(5):
        # Create a 3D scatter plot
        fig = plt.figure()
        ax = fig.add_subplot(111, projection='3d')
        for bodyparts in config.visualization['skeleton'][::-1]:
            idx_bodyparts = []
            for bodypart in bodyparts:
                idx_bodyparts.append(config.animal['bodyparts'].index(bodypart))
            ax.plot(rotated_points_3d[idxs[i,j],idx_bodyparts,0],rotated_points_3d[idxs[i,j],idx_bodyparts,1],rotated_points_3d[idxs[i,j],idx_bodyparts,2], marker = 'o', ms=3)
        ax.set_title(f'Cluster {i}, frame {idxs[i,j]}')
        # ax.set_xlim((0,xlim))
        # ax.set_ylim((0,ylim))
        # ax.set_zlim((0,zlim))

In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

xlim, ylim, zlim, _ = config.visualization['room_dimensions']
normalized_velocity = rotated_points_3d_velocity_3d/np.sqrt(np.einsum("ijkl,ijkl->j",vector_body,vector_body)[:,np.newaxis,np.newaxis])[:-1]*100
normalized_velocity = np.concatenate((normalized_velocity,np.mean(normalized_velocity,axis=1,keepdims=True)),axis=1)
normalized_velocity = np.concatenate((normalized_velocity,np.sqrt(np.einsum("ijk,ijk->ij", normalized_velocity, normalized_velocity))[:,:,np.newaxis]),axis=2)

normalized_rotated_points_3d = rotated_points_3d[:-1]/np.sqrt(np.einsum("ijkl,ijkl->j",vector_body,vector_body)[:,np.newaxis,np.newaxis])[:-1]*100

for i in range(np.unique(clusters)):
    idx_cluster_i = np.nonzero(clusters == i)[0]
    mean_velocity = np.mean(normalized_velocity[idx_cluster_i,:,:], axis=0).transpose((1,0))
    var_velocity = np.var(normalized_velocity[idx_cluster_i,:,:], axis=0).transpose((1,0))

    fig, ax = plt.subplots(figsize=(16, 4))
    ax.axis('off')
    norm = mcolors.Normalize(vmin=0, vmax=5)
    cmap = plt.cm.Reds
    table = ax.table(
        cellText=np.round(mean_velocity,decimals=2),
        cellColours= cmap(norm(var_velocity)),
        rowLabels=['Velocity X','Velocity Y','Velocity Z','Velocity magnitude'],
        colLabels=config.animal['bodyparts'] + ['Mean'],
        loc='center',
        cellLoc='center',
        fontsize = 20
    )

    mean_points_3d = np.mean(normalized_rotated_points_3d[idx_cluster_i,:,:], axis=0)

    # Create a 3D scatter plot
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')
    for bodyparts in config.visualization['skeleton'][::-1]:
        idx_bodyparts = []
        for bodypart in bodyparts:
            idx_bodyparts.append(config.animal['bodyparts'].index(bodypart))
        ax.plot(mean_points_3d[idx_bodyparts,0],mean_points_3d[idx_bodyparts,1],mean_points_3d[idx_bodyparts,2], marker = 'o', ms=3)
    ax.set_title(f'Cluster {i}, Average')
    # ax.set_xlim((0,xlim))
    # ax.set_ylim((0,ylim))
    # ax.set_zlim((0,zlim))
        